In [ ]:
%cd ../..

import os
import sys

import requests  # For making HTTP requests
import pandas as pd  # For data manipulation
import sqlalchemy  # For interacting with databases
from sqlalchemy.engine import URL, create_engine
from shapely import wkb
import numpy as np  # For numerical operations (if needed)
import matplotlib.pyplot as plt  # For visualizing data (if needed)
import seaborn as sns  # For data visualization with Seaborn (if needed)
import plotly.express as px  # For data visualization with Plotly (if needed)
import geopandas as gpd
from matplotlib.dates import HourLocator, DateFormatter

# Custom imports
# Add the path to the directory containing your local module

from utils.utilities import run_sql

In [ ]:
# trips are stored in anon_master (this is just a join of (materialized) views anon_*
df_trips = run_sql('trips', index_col='track_id')

df_trips

In [ ]:
# fleet information
df_fleet = run_sql('fleet', index_col='vehicle_id')

vehicle_to_ff = df_fleet['fleet_test_id'].to_dict()
df_trips['freight_forwarder'] = df_trips['vehicle_id'].map(vehicle_to_ff)

df_trips.to_csv('input/stations/tracks.csv')
df_fleet.to_csv('input/home/fleet.csv')
df_fleet.head()

In [ ]:
# track_id map information

#alternative shorter version, without checking if index and track_id_new match
#df_track_ids = run_sql('track_ids', index_col='track_id_new')

df_track_ids = run_sql('track_ids')
assert all(df_track_ids.index == df_track_ids['track_id_new']), 'Index does not match track_id_new'
df_track_ids.set_index('track_id_new', inplace=True)

df_track_ids.to_csv('input/stations/track_ids.csv')
df_track_ids.head()

## Remnants from the Spirit-e repository. The following code does not have to be excecuted 

In [ ]:
# TODO maybe implement the collection of the reduced tracks for energy simulation here

# SQL query to get the detailed GPS data for a specific track_id
# select * from ftm_gps_lon_lat_from_track_id ({trackid})

detailed_tracks = run_sql_query('select * from ftm_gps_lon_lat_from_track_id (1500000552337)')

#track_gps = run_sql_query('select * from ftm_gps_lon_lat_from_track_id(25)', index_col="track_id")

In [ ]:
#  # trips are stored in anon_master (this is just a join of (materialized) views anon_*
# df_nuts3_trips = pd.read_sql('''select t.fid, n, ST_Transform(nm.geom, 3857) as geom from 
# (
# select substring(anz.fid,0,6) as fid, sum(count) as n from nefton_rio_telemetry.anon_nuts_zones anz
# group by substring(anz.fid,0,6)
# order by n desc
# ) t
# join administrative_boundaries.nuts_2021_10m nm on t.fid  = nm.fid
# where n > 0
# order by n desc''', con=engine, index_col='fid')
# # TODO Why is the db nefton_rio_telemetry used here? Shouldn't it be spirite? Should I do the same?
# # sql script for the creation of nefton_rio_telemetry is in old repo on LRZ syncandshare

# df_nuts0 = pd.read_sql('''select gid, fid, ST_Transform(nm.geom, 3857) as geom from 
# administrative_boundaries.nuts_2021_10m nm where levl_code = 0''', con=engine, index_col='gid')

# df_nuts1 = pd.read_sql('''select gid, fid, ST_Transform(nm.geom, 3857) as geom from 
# administrative_boundaries.nuts_2021_10m nm where levl_code = 1''', con=engine, index_col='gid')

# gdf_nuts3_trips = gpd.GeoDataFrame(df_nuts3_trips[['n']], geometry=df_nuts3_trips.geom.apply(lambda x: wkb.loads(x, hex=True)))

# gdf_nuts0 = gpd.GeoDataFrame(df_nuts0[['fid']], geometry=df_nuts0.geom.apply(lambda x: wkb.loads(x, hex=True)))
# gdf_nuts1 = gpd.GeoDataFrame(df_nuts1[['fid']], geometry=df_nuts1.geom.apply(lambda x: wkb.loads(x, hex=True)))



# #TODO: Für Heatmap nutzen und Gedanken dazu machen, wie man das anonymisieren könnte

# # locations - not to be published only for the heat map!
# df_start_locations = pd.read_sql('SELECT track_id, start_location from spirite.anon_layer_base', con=engine, index_col='track_id')
# df_start_locations['start_location'] = df_start_locations.start_location.apply(lambda x: wkb.loads(x, hex=True))
# df_start_locations.to_pickle('input/df_start_locations.pkl')
# df_start_locations.head()